# Generate Synthetic Wrought Alloy Database

Uses GMM (tuned in 02) to sample compositions and per-target forward models (from config) to predict properties. Saves **synthetic_wrought.csv** as the search pool for 05_backward_wrought. Run 01 and 02 first.

In [1]:
import pandas as pd
import numpy as np
import os
import xgboost as xgb
from sklearn.mixture import GaussianMixture
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
import warnings

warnings.filterwarnings('ignore')

try:
    from utils import load_hyperparams, load_hyperparams_for_target, load_backward_gmm_params, get_default_hyperparams
except ImportError:
    import sys
    sys.path.insert(0, os.getcwd())
    from utils import load_hyperparams, load_hyperparams_for_target, load_backward_gmm_params, get_default_hyperparams

INPUT_COLS = ['Al', 'Si', 'Fe', 'Cu', 'Mn', 'Mg', 'Cr', 'Ni', 'Zn', 'Ga', 'V', 'Ti']
WROUGHT_PATH = 'wrought_alloys_final.csv'
OUTPUT_PATH = 'synthetic_wrought.csv'
N_SAMPLES = 50000

In [2]:
# Load wrought data (full df with compositions + targets for training models)
def load_wrought(path):
    if path.endswith('.xlsx') or path.endswith('.xls'):
        df = pd.read_excel(path)
    else:
        try:
            with open(path, 'rb') as f:
                head = f.read(2)
            if head == b'PK':
                df = pd.read_excel(path)
            else:
                try:
                    df = pd.read_csv(path, encoding='utf-8')
                except UnicodeDecodeError:
                    df = pd.read_csv(path, encoding='latin-1')
        except Exception:
            df = pd.read_excel(path)
    exclude = INPUT_COLS + ['Series', 'Parent Alloy', 'Alloy Name', 'AlloyNumber', 'Temper']
    targets = [c for c in df.columns if c not in exclude]
    for col in INPUT_COLS + targets:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    df[INPUT_COLS] = df[INPUT_COLS].fillna(0.0)
    return df, targets

df_real, target_list = load_wrought(WROUGHT_PATH) if os.path.exists(WROUGHT_PATH) else (None, [])
if df_real is None:
    raise FileNotFoundError(f'Not found: {WROUGHT_PATH}. Run with wrought data.')
print(f'Loaded wrought data: {df_real.shape}, targets: {len(target_list)}')

Loaded wrought data: (868, 31), targets: 14


In [3]:
# Fit GMM on real compositions and sample
gmm_params = load_backward_gmm_params('wrought') or get_default_hyperparams('GMM')
X_real = df_real[INPUT_COLS]
gmm = GaussianMixture(**gmm_params)
gmm.fit(X_real)
samples, _ = gmm.sample(N_SAMPLES)
gen_df = pd.DataFrame(samples, columns=INPUT_COLS)
gen_df[gen_df < 0] = 0
gen_df['_sum'] = gen_df.sum(axis=1)
gen_df = gen_df[gen_df['_sum'] > 0.1]
for c in INPUT_COLS:
    gen_df[c] = (gen_df[c] / gen_df['_sum']) * 100
gen_df = gen_df.drop(columns=['_sum']).fillna(0.0).reset_index(drop=True)
print(f'Sampled {len(gen_df)} valid compositions.')

Sampled 50000 valid compositions.


In [4]:
# Build per-target models from config, train on real data, predict on synthetic
def build_model(name, params):
    p = params.copy() if params else get_default_hyperparams(name)
    if name == 'XGBoost':
        return xgb.XGBRegressor(objective='reg:squarederror', **{k: v for k, v in p.items()})
    if name == 'RandomForest':
        return RandomForestRegressor(**{k: v for k, v in p.items()})
    if name == 'GradientBoosting':
        return GradientBoostingRegressor(**{k: v for k, v in p.items()})
    return None

wrought_config = load_hyperparams('wrought')
by_target = (wrought_config or {}).get('by_target') or {}
if not by_target:
    raise ValueError('wrought.by_target not in config. Run 01_hyperparameter_tuning_forward.ipynb first.')

for target, spec in by_target.items():
    if target not in df_real.columns or df_real[target].notna().sum() < 10:
        continue
    model_name = spec.get('model')
    params = spec.get('params')
    model = build_model(model_name, params)
    if model is None:
        continue
    df_t = df_real.dropna(subset=[target])
    X_train = df_t[INPUT_COLS]
    y_train = df_t[target]
    model.fit(X_train, y_train)
    gen_df[target] = model.predict(gen_df[INPUT_COLS])
    print(f'  Predicted {target[:45]}')

print(f'Total columns: {list(gen_df.columns)}')

  Predicted UTS (MPa)


  Predicted YS (MPa)


  Predicted El (%)


  Predicted Fatigue Strength (MPa)


  Predicted Shear Strength (MPa)


...

  Predicted TC (W/m-K)


  Predicted TE Coeff


  Predicted Thermal Diffusivity 


  Predicted EC Volume (% IACS)


  Predicted EC Weight (% IACS)
Total columns: ['Al', 'Si', 'Fe', 'Cu', 'Mn', 'Mg', 'Cr', 'Ni', 'Zn', 'Ga', 'V', 'Ti', 'UTS (MPa)', 'YS (MPa)', 'El (%)', 'Fatigue Strength (MPa)', 'Shear Strength (MPa)', 'Y (GPa)', 'G (GPa)', 'Density (g/cc)', 'Cp (J/kg-K)', 'TC (W/m-K)', 'TE Coeff', 'Thermal Diffusivity ', 'EC Volume (% IACS)', 'EC Weight (% IACS)']


In [5]:
# Save synthetic pool for backward search
gen_df.to_csv(OUTPUT_PATH, index=False)
print(f'Saved {len(gen_df)} rows to {OUTPUT_PATH}')

Saved 50000 rows to synthetic_wrought.csv
